In [0]:
import dlt
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
@dlt.table(
    name="maven_catalog.bronze_schema.brz_products_mongo_dlt",
    comment="Products from MongoDB Atlas with data quality checks",
    table_properties={
        "quality": "bronze",
        "source": "mongodb_atlas"
    }
)
# Note: Ensure these column names match the landing table casing exactly
@dlt.expect("valid_product_id", "product_id IS NOT NULL")
@dlt.expect("valid_product_name", "product_name IS NOT NULL")
@dlt.expect("valid_retail_price", "product_retail_price > 0")
@dlt.expect("valid_cost", "product_cost > 0")
@dlt.expect("valid_sku", "product_sku IS NOT NULL")
def brz_products_mongo_dlt():
    # Use readStream for incremental ingestion from the landing table
    return (
        spark.readStream.table("maven_catalog.maven_market_landing.products")
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit("mongodb_atlas"))
    )

In [0]:
@dlt.table(
    name= "maven_catalog.bronze_schema.brz_customers_mongo_dlt",
    comment="Customers from MongoDB Atlas with data quality checks",
    table_properties={
        "quality": "bronze",
        "source": "mongodb_atlas"
    }
)
@dlt.expect("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect("valid_first_name", "first_name IS NOT NULL")
@dlt.expect("valid_last_name", "last_name IS NOT NULL")
@dlt.expect("valid_country", "customer_country IS NOT NULL")
@dlt.expect("valid_gender", "gender IN ('M', 'F')")
def brz_customers_mongo_dlt():
    # Use readStream to ensure CDC (Change Data Capture) works for your SCD2 Silver layer
    return (
        spark.readStream.table("maven_catalog.maven_market_landing.customers")
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit("mongodb_atlas"))
    ) 